In [1]:
import torch
import torch.nn as nn
import tiktoken

from DecoderComponents import LayerNorm
from TransformerBlock import TransformerBlock

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [ ]:
class GPTModel(nn.Module):
    ''' 
    A final GPT model created after collecting all GPT Model
    '''
    def __init__(self, cfg:dict):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")
batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [5]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 5.8380e-01, -1.0901e-01, -1.0397e+00,  ...,  1.5072e-02,
          -2.1667e-01, -4.7530e-01],
         [ 2.0323e-01, -2.8204e-01, -8.5981e-01,  ..., -1.9883e-01,
          -8.5702e-04, -3.5104e-01],
         [ 9.0067e-01,  1.5839e-01, -6.5688e-01,  ...,  2.4166e-02,
          -9.1903e-01, -1.8686e-01],
         [-1.1552e+00,  8.3347e-01, -4.3708e-01,  ...,  8.2582e-01,
           2.1277e-01, -4.4075e-01]],

        [[ 2.0957e-02, -4.8953e-01, -5.8782e-01,  ...,  3.3615e-01,
           1.2045e-02, -7.8100e-01],
         [ 1.0735e-01, -3.4241e-02, -1.5861e-01,  ...,  4.1621e-01,
          -2.7504e-01,  3.1481e-01],
         [ 1.1170e+00,  7.1822e-01,  2.7271e-02,  ...,  5.3295e-01,
          -7.6368e-02,  9.0427e-02],
         [ 1.0473e-01,  3.8360e-01,  5.4795e-01,  ...,  1.1498e+00,
          -2.5496e-01, -7.9190e-02]]], grad_fn=<UnsafeViewBackward0>)


In [6]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


In [7]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


In [8]:
total_params_gpt2 = (total_params - sum(p.numel()
    for p in model.out_head.parameters()))
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

Number of trainable parameters considering weight tying: 124,412,160


In [9]:
total_size_bytes = total_params * 4
total_size_mb = total_size_bytes / (1024 * 1024)
print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


In [10]:
# A function for the GPT model to generate text

# Crops current context if it exceeds the supported context size,
# e.g., if LLM supports only 5 tokens, and the context size is 10,
# then only the last 5 tokens are used as context

def generate_text_simple(model, idx, max_new_token, context_size):
    for _ in range(max_new_token):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim = 1)

    return idx

In [11]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [12]:
model.eval()
out = generate_text_simple(model=model, idx = encoded_tensor, max_new_token=6, context_size=GPT_CONFIG_124M["context_length"])
print("Output: ", out)
print("Output length: ", len(out[0]))

Output:  tensor([[15496,    11,   314,   716, 40179,   620, 29604, 24432, 34562, 42179]])
Output length:  10


In [13]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am677ach remediesgroups bourgeoisie risked
